# 09 — Report: every model, one table

Step 9 of the protocol (D105), and the last. No GPU, and no model is loaded. Each run of notebooks
03, 06 and 07 left its scores, its transcripts and its per-clip error counts on the hub; this
notebook reads them back and lays them side by side: base Flex, every Flex run and blend, the
frozen teacher, and every student, ASR and speech-LLM, at every stage.

**Two families of students.** The ASR students (06) and the speech-LLM students (07, D122) share
the recipe, the data and the evaluation; the report labels every run with its family
(`evalkit.family`) and reads the recipe's gain per family, which is the claim the paper makes
(D121), not which family wins.

**Why the numbers can sit in one table.** Every run was scored by the same code (`evalkit`), on
the same export (`DATASET_EXPORT`, checked by each notebook before it trained or scored), with the
same fold, and each model card records the export, the fold version and the kits' digests. A run
scored on another export is left out and named.

**Pairings are taken here**, from the per-clip counts, on the clips both runs scored, with
episodes resampled (`evalkit.pair`). So any two models can be compared, not only the pairs a
training notebook thought of.

**The success bar for a student** (roadmap §B): its interval against the teacher contains zero, or
lies below it, on gold and val, class by class. Read the crosstalk classes with care: clips
overlapped beyond `MAX_OVERLAP_SHARE` were kept out of the pseudo-labels in step 5.

**Gold is held out.** Nothing in the protocol is chosen on gold or on the public sets: recipes,
blends and the teacher are chosen on val. The gold and public columns are scores, not selections.

**Outputs**: `FLEX_REPO/<flex prefix>/report.json` and `report.md` (`report-smoke.*` from a smoke
run, which reads the real Flex runs when there is no smoke teacher).

## Config

In [ ]:
NOTEBOOK = "09_Report"
DATASET_REPO = "Sagyam/nepanglish-asr"
DATASET_EXPORT = "2026-09-30"        # runs scored on another export are left out
FLEX_REPO = "Sagyam/nepanglish-asr-flex-ft"      # teacher.json names the Flex runs' folder
STUDENTS_REPO = "Sagyam/nepanglish-asr-students"
STUDENTS_PREFIX = "students-2026-09-30"
SMOKE = False             # True reads the -smoke folders, and teacher-smoke.json if there is one
CLASSES = ("overlap", "cmi")   # the clip classes every pairing is also split by
# Student A minus student B at every stage both have (D122): the LLM decoder against the CTC head on
# one encoder family, and each speech-LLM student against Whisper, the ASR student to beat (D121).
CROSS = (("omni-llm", "omnilingual"), ("gemma-e2b", "whisper"), ("omni-llm", "whisper"))

## Setup

In [ ]:
%pip install -q numpy
import json
import os
import sys
from pathlib import Path

# Before torch touches CUDA: lets the allocator grow segments instead of fragmenting when batch
# shapes vary (a fresh kernel only).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_COLAB = "google.colab" in sys.modules
# Secrets only work from a cell run in the Colab UI. When cells are driven from outside (the Colab
# MCP), run this cell by hand once. The token is then kept in the hub's own token file on the VM
# (where `hf auth login` puts it), so a kernel restart needs no click; the file goes when the
# runtime is deleted.
if IN_COLAB:
    from google.colab import userdata

    TOKEN_FILE = Path.home() / ".cache" / "huggingface" / "token"
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = (TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists()
                                  else userdata.get("HF_TOKEN"))
    if not TOKEN_FILE.exists():
        TOKEN_FILE.parent.mkdir(parents=True, exist_ok=True)
        TOKEN_FILE.write_text(os.environ["HF_TOKEN"])
        TOKEN_FILE.chmod(0o600)
FT = Path("/content/ft") if IN_COLAB else Path.cwd() / ".cache-ft"
FT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(FT))
if SMOKE:
    STUDENTS_PREFIX += "-smoke"

In [ ]:
%%writefile /content/ft/evalkit.py
"""One evaluation for every model (D105).

Whatever was trained -- Flex, a blend of its weights, a student at any stage -- is scored by the
functions here, so a number in one notebook means what it means in another:

  * gold and val, folded and raw, split into substitutions, deletions and insertions, overall and
    per clip class, and paired clip by clip against named reference runs with episodes resampled
    (`evaluate_run`);
  * the public Nepali sets, folded, raw and plain, paired against base Flex with each set's own
    unit resampled (`run_benchmarks`).

Every split and set also keeps its errors: each aligned word pair, classified, in
`harness/errors/<set>.parquet` with each clip's crosstalk and SNR, and a `breakdown` beside its
WER (crosstalk buckets, numbers, the
most common substitutions, deletions and insertions; D110) and, given the training corpus's word
counts, the WER within each bucket of how often a word was trained on (D123). Both come from the
harness's own `error_mining.py` and `error_store.py`, which the dataset's `harness/` copy carries
beside fold.py, so the Models page reads exactly what the notebook wrote.

The notebook hands in what is model-specific: `decode`, its batched decoder, and `score`,
`ftkit.harness_scorer`, which is the harness's own fold.py. No torch here, so the rules are tested
from backend/tests. Keep it importable on Python 3.10+.
"""

from __future__ import annotations

import datetime as dt
import hashlib
import io
import json
import shutil
import subprocess
import tarfile
import time
import unicodedata
from collections.abc import Callable, Collection, Iterator, Mapping, Sequence
from contextlib import contextmanager
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import distill
import sweep

SR = 16_000
#: Scored in this order: gold first, so a run stopped while scoring has the held-out number.
SPLITS = ("gold", "val")
BUCKETS = ("none", "0-5%", "5-15%", ">15%")
#: The clip classes every paired difference is also split by (D87).
REPORT_KEYS = ("overlap", "snr", "speakers", "cmi", "duration")
#: A class with fewer clips is left out of a paired difference: resampling one or two clips
#: prints an interval of +-60 or +-100, which is noise, not a finding.
MIN_CLASS_CLIPS = 10
#: What a result row keeps of a split's metrics.
KEEP = ("wer", "raw_wer", "cer", "sub", "del", "ins", "loops", "clips", "rtf", "vs")

#: The students whose decoder is an LLM (07a, 07b; D122). Every other student is an ASR model.
SPEECH_LLM_STUDENTS = ("gemma-e2b", "omni-llm")

#: `decode(rows)` -> texts, per-clip compute seconds, and the loop retries as
#: (segment_id, first decode, retried decode).
Decode = Callable[[Sequence[dict]], tuple[list[str], list[float], list[tuple[str, str, str]]]]
Counts = Mapping[str, Sequence[int]]


@contextmanager
def timed(label: str) -> Iterator[None]:
    """`label...` before a step and `label: done in N s` after it: a stalled step names itself."""
    print(f"{label}...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"{label}: done in {time.perf_counter() - t0:.0f} s", flush=True)


def duration(row: dict) -> float:
    return row["end_time"] - row["start_time"]


# --- provenance ----------------------------------------------------------------------------------


def family(row: Mapping[str, Any]) -> str:
    """The family of a run: "Flex" for one with no student (base, the Flex runs and blends, the
    teacher), else its student's, "speech-LLM" or "ASR". The report reads the recipe's gain per
    family (D121)."""
    student = row.get("student")
    if not student:
        return "Flex"
    return "speech-LLM" if student in SPEECH_LLM_STUDENTS else "ASR"


def check_export(manifest: Mapping[str, Any], expected: str) -> None:
    """Refuse a dataset that is not the export every model in the comparison trains on.

    `expected` is the start of the training manifest's `exported_at` ("2026-09-30"). A date, not a
    commit: the dataset repo's history is squashed after each upload, which would orphan a pinned
    commit, and the export's own timestamp survives it."""
    got = str(manifest.get("exported_at") or "")
    if not expected or not got.startswith(expected):
        raise ValueError(
            f"the dataset is the {got or 'undated'} export, not {expected!r}: every model in one "
            "comparison trains and is scored on the same export"
        )


def kit_digests(folder: Path) -> dict[str, str]:
    """The first 16 hex digits of each kit file's sha256, for the model card: which code ran."""
    return {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest()[:16]
        for p in sorted(Path(folder).glob("*.py"))
    }


# --- transcripts and per-clip counts -------------------------------------------------------------


def read_hyps(path: Path | str) -> dict[str, str]:
    """A transcript file (`write_hyps`) as {segment_id: text}."""
    with Path(path).open(encoding="utf-8") as fh:
        return {j["segment_id"]: j["text"] for j in map(json.loads, fh)}


def write_hyps(
    path: Path, rows: Sequence[dict], texts: Sequence[str], compute: Sequence[float]
) -> None:
    """One transcript per line, the format the harness's Models page imports (D83)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        for r, text, c in zip(rows, texts, compute, strict=True):
            row = {"segment_id": r["segment_id"], "text": text, "compute_s": c}
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")


def counts(rows: Sequence[dict], clips: Sequence[Mapping[str, int]]) -> dict[str, list[int]]:
    """{segment_id: [folded errors, reference words]}: all a paired comparison needs of a run."""
    return {r["segment_id"]: [c["errors"], c["words"]] for r, c in zip(rows, clips, strict=True)}


def reference_counts(rows: Sequence[dict], by_id: Mapping[str, str], score: Any) -> dict:
    """Another system's transcripts scored against these rows' labels, on the clips it has.

    For a reference that left only transcripts (a run from an earlier export): its counts are
    taken against the current labels, and `pair` compares on the clips both scored."""
    have = [r for r in rows if r["segment_id"] in by_id]
    clips = score.per_clip([r["text"] for r in have], [by_id[r["segment_id"]] for r in have])
    return counts(have, clips)


def pair(
    rows: Sequence[dict],
    a: Counts,
    b: Counts,
    *,
    keys: Sequence[str] = REPORT_KEYS,
    group: str = "episode_id",
    n: int = 2000,
    min_clips: int = MIN_CLASS_CLIPS,
) -> dict[str, Any]:
    """WER(b) - WER(a) in points with a 95% interval, on the clips both runs scored.

    `group` names the unit resampled: a clip's episode on gold and val, a speaker, sentence or
    video on a public set. Returns `clips` (how many were shared), `all`, and one entry per value
    of each clip class in `keys` holding at least `min_clips` shared clips, as
    `[difference, low, high]`. Raises when no clip is shared."""
    idx = [i for i, r in enumerate(rows) if r["segment_id"] in a and r["segment_id"] in b]
    if not idx:
        raise ValueError("the two runs share no clip")

    def diff(which: Sequence[int]) -> list[float]:
        return list(
            sweep.paired_bootstrap(
                [
                    dict(zip(("errors", "words"), a[rows[i]["segment_id"]], strict=True))
                    for i in which
                ],
                [
                    dict(zip(("errors", "words"), b[rows[i]["segment_id"]], strict=True))
                    for i in which
                ],
                [str(rows[i][group]) for i in which],
                n=n,
            )
        )

    out: dict[str, Any] = {"clips": len(idx), "all": diff(idx)}
    for key in keys:
        values: dict[str, list[int]] = {}
        for i in idx:
            value = (rows[i].get("classes") or {}).get(key)
            if value is not None:
                values.setdefault(str(value), []).append(i)
        for value, which in sorted(values.items()):
            if len(which) >= min_clips:
                out[f"{key}={value}"] = diff(which)
    return out


# --- error mining (D110) -------------------------------------------------------------------------


def _miner() -> tuple[Any, Any] | None:
    """`error_mining` and `error_store` from the harness copy `ftkit.harness_scorer` put on the
    path, or None -- with a line saying so -- when the dataset's `harness/` predates them."""
    try:
        from app.services import error_mining, error_store
    except ImportError as exc:
        print(f"no error mining in the dataset's harness/ ({exc}): no error files are written")
        return None
    return error_mining, error_store


def _share(row: Mapping[str, Any]) -> float | None:
    """A gold or val clip's crosstalk share: its own, or from the export's `overlap_spans`."""
    if "overlap_share" in row:
        return row["overlap_share"]
    return distill.overlap_share(row.get("overlap_spans"), duration(row))


def write_errors(
    path: Path,
    run: str,
    rows: Sequence[dict],
    texts: Sequence[str],
    clips: Sequence[Any],
    *,
    overlap: Mapping[str, float | None] | None = None,
    snr: Mapping[str, float | None] | None = None,
    by: str | None = None,
    word_counts: Path | None = None,
) -> dict[str, Any] | None:
    """Every aligned word pair of one split or set, classified, written to `path` (its stem is
    the set's name); returns the file's report (blocks 1-3, and `rarity` with `word_counts`), or
    None when nothing was written.

    A clip's alignment is the one `score.per_clip` counted, when it kept it. `overlap` and `snr`
    are a public set's measured crosstalk and speech-to-noise ratio by clip id; gold and val rows
    carry their own `acoustics`, and their crosstalk as the export's clip-relative
    `overlap_spans` (or an `overlap_share` already taken from them)."""
    kit = _miner()
    if kit is None:
        return None
    mining, store = kit
    ids = mining.unique_ids(str(r["segment_id"]) for r in rows)
    found = mining.rows(
        run,
        Path(path).stem,
        [
            {
                "clip_id": r["segment_id"],
                "group": r["episode_id"],
                "ref": r["text"],
                "hyp": text,
                "overlap_share": overlap.get(i) if overlap is not None else _share(r),
                "snr_db": snr.get(i)
                if snr is not None
                else (r.get("acoustics") or {}).get("snr_db"),
                "by": r.get(by) if by else None,
                "alignment": c.get("alignment") if isinstance(c, Mapping) else None,
            }
            for r, text, c, i in zip(rows, texts, clips, ids, strict=True)
        ],
    )
    if not found:
        return None
    mining.write(found, path)
    report = store.report(Path(path))
    if word_counts is not None:
        report["rarity"] = rarity(Path(path), word_counts)
    return report


# --- word rarity (D123) --------------------------------------------------------------------------

#: The training corpus's word counts, in the dataset's harness/ copy beside the scorer: every
#: notebook's `ftkit.download_dataset` already brings it. 05_Teacher writes it with its labels.
WORD_COUNTS_FILE = "harness/word_counts.json"


def word_counts_path(data: Path) -> Path | None:
    """The dataset's word counts, or None -- with a line saying so -- when it has none yet (no
    teacher has labelled anything): the runs are then scored without their rarity block, and
    `add_rarity` adds it later."""
    path = Path(data) / WORD_COUNTS_FILE
    if not path.exists():
        print(f"no {WORD_COUNTS_FILE} in the dataset: errors are not split by word rarity")
        return None
    return path


def count_training_words(
    path: Path,
    train: Sequence[Mapping[str, Any]],
    pseudo: Sequence[Mapping[str, Any]],
    *,
    export: str,
    labels: str,
) -> Path:
    """Count every word of what the students train on -- stage 1's human train labels and stage
    2's pseudo-labels -- by spelling key, and write the counts to `path` (D123).

    `export` is the export's `exported_at` and `labels` the pseudo-labels' folder: the file names
    both, so a bucket can be traced to the texts it was counted on."""
    kit = _miner()
    if kit is None:
        raise RuntimeError("the dataset's harness/ has no error mining: upload the harness copy")
    mining, _ = kit
    texts = [r["text"] for r in train] + [r["text"] for r in pseudo]
    sources = {
        "human": {"export": export, "clips": len(train)},
        "pseudo": {"labels": labels, "clips": len(pseudo)},
    }
    mining.write_counts(path, mining.word_counts(texts), sources=sources)
    return Path(path)


def rarity(errors: Path, word_counts: Path) -> dict[str, Any] | None:
    """An error file's WER within each word-rarity bucket (`error_store.rarity`), with the corpus
    it was counted on; None when the harness copy predates it."""
    kit = _miner()
    if kit is None or not hasattr(kit[1], "rarity"):
        print("the dataset's harness/ predates word rarity (D123): upload the harness copy")
        return None
    mining, store = kit
    meta, counts = mining.read_counts(word_counts)
    keep = ("fold_version", "sources", "types", "tokens", "created_at")
    return store.rarity(Path(errors), counts) | {"corpus": {k: meta.get(k) for k in keep}}


def add_rarity(out: Path, word_counts: Path) -> dict[str, dict]:
    """Add the rarity block to a finished run's breakdowns, from its error files: each
    `harness/errors/<set>.parquet` is read, and `<split>_metrics.json` or `benchmarks/<set>.json`
    gains `breakdown.rarity`. Nothing is decoded or scored again. Returns {set: block}."""
    out = Path(out)
    added = {}
    for errors in sorted((out / "harness" / "errors").glob("*.parquet")):
        name = errors.stem
        summary = (
            out / f"{name}_metrics.json" if name in SPLITS else out / "benchmarks" / f"{name}.json"
        )
        if not summary.exists():
            print(f"{name}: an error file but no {summary.name}; left as it is")
            continue
        block = rarity(errors, word_counts)
        if block is None:
            break
        data = json.loads(summary.read_text("utf-8"))
        data["breakdown"] = {**(data.get("breakdown") or {}), "rarity": block}
        summary.write_text(json.dumps(data, indent=1, ensure_ascii=False))
        added[name] = block
        print_rarity(name, block)
    return added


def print_rarity(label: str, block: Mapping[str, Any] | None) -> None:
    """The rarity block in one line: each bucket's WER and its share of the reference words."""
    if not block:
        return
    parts = [
        f"{b['bucket']} {'-' if b['wer'] is None else format(b['wer'], '.2f')}"
        f" ({100 * b['share_of_words']:.1f}% of words)"
        for b in block["buckets"]
    ]
    print(f"    {label} WER by word rarity: " + " · ".join(parts))


#: A public set's measured recording conditions, each `benchmarks/<kind>/<set>.parquet`.
CONDITIONS = ("overlap", "acoustics")


def fetch_conditions(
    repo: str, token: str, folder: Path, names: Sequence[str] | None = None
) -> Path:
    """Each public set's measured crosstalk and acoustics, `benchmarks/{overlap,acoustics}/<set>
    .parquet` in `repo`, copied into `folder/overlap/` and `folder/acoustics/`
    (scripts/measure_benchmark_overlap.py wrote them); a set without one is unmeasured on it."""
    from huggingface_hub import hf_hub_download
    from huggingface_hub.utils import EntryNotFoundError

    folder = Path(folder)
    for kind in CONDITIONS:
        (folder / kind).mkdir(parents=True, exist_ok=True)
        for name in names or tuple(BENCHMARKS):
            try:
                got = hf_hub_download(repo, f"benchmarks/{kind}/{name}.parquet", token=token)
            except EntryNotFoundError:
                print(f"{name}: no measured {kind} in {repo}; its clips are unmeasured on it")
                continue
            shutil.copyfile(got, folder / kind / f"{name}.parquet")
    return folder


def print_breakdown(label: str, report: Mapping[str, Any] | None) -> None:
    """Block 1 and block 2 in two lines, under a WER line."""
    if not report:
        return
    buckets = " · ".join(
        f"{b['bucket']} {b['wer']:.2f} ({100 * b['share_of_errors']:.0f}% of errors)"
        for b in report["overlap"]
    )
    n = report["numbers"]
    print(f"    {label} crosstalk: {buckets}")
    print(
        f"    {label} numbers: {n['errors']} errors ({100 * n['share_of_errors']:.1f}%), "
        f"WER without them {n['wer_without']:.2f}"
    )
    print_rarity(label, report.get("rarity"))


# --- gold and val --------------------------------------------------------------------------------


def score_split(
    rows: Sequence[dict],
    texts: Sequence[str],
    compute: Sequence[float],
    log: Sequence[tuple[str, str, str]],
    score: Any,
    *,
    errors: Path | None = None,
    run: str = "",
    word_counts: Path | None = None,
) -> tuple[dict[str, Any], dict[str, list[int]]]:
    """One split's metrics and its per-clip counts, each clip aligned once.

    Metrics: the folded, raw and character error rates with S/D/I, the real-time factor, the loop
    retries and the greedy-only score from the same decode, and every clip class (`by_class`).
    With `errors` (`harness/errors/<split>.parquet`), the split's error rows are written there and
    its `breakdown` added, split by word rarity with `word_counts` (`count_training_words`)."""
    refs = [r["text"] for r in rows]
    clips = score.per_clip(refs, texts)
    m = score.summarize(clips)
    m["rtf"] = sum(compute) / max(sum(duration(r) for r in rows), 1e-9)
    m["retried"] = [{"segment_id": s, "first": f, "retry": t} for s, f, t in log]
    if log:  # the same run before any retry: only the retried clips are aligned again
        first = {s: f for s, f, _ in log}
        idx = [i for i, r in enumerate(rows) if r["segment_id"] in first]
        again = score.per_clip([refs[i] for i in idx], [first[rows[i]["segment_id"]] for i in idx])
        greedy = list(clips)
        for i, c in zip(idx, again, strict=True):
            greedy[i] = c
        m["greedy_only"] = score.summarize(greedy)
    m["by_class"] = distill.by_class(rows, clips, score.summarize)
    if errors is not None:
        m["breakdown"] = write_errors(errors, run, rows, texts, clips, word_counts=word_counts)
    return m, counts(rows, clips)


def evaluate_run(
    out: Path,
    run: str,
    *,
    splits: Mapping[str, Sequence[dict]],
    decode: Decode,
    score: Any,
    card: Mapping[str, Any],
    meta: Mapping[str, Any] | None = None,
    references: Mapping[str, Mapping[str, Counts]] | None = None,
    keys: Sequence[str] = REPORT_KEYS,
    word_counts: Path | None = None,
) -> dict[str, Any]:
    """Decode gold and val with the weights `decode` reads, score them, and write the run's files.

    Written under `out`: `harness/<split>.jsonl` and `harness/model_card.json` (what the Models
    page imports, D83), `harness/errors/<split>.parquet` (its error rows), `<split>_metrics.json`,
    `per_clip.json` (the counts another run is paired against) and `result.json`, whose presence
    on the hub marks the run as done. `references` is {name: {split: per-clip counts}}; each
    becomes `vs[name]`, this run minus that one. `meta` (recipe, seed, stage, best epoch, ...) goes
    into the card and the result row, which is returned. `word_counts` (`word_counts_path`) adds
    each split's WER by word rarity to its breakdown."""
    out = Path(out)
    results: dict[str, dict] = {}
    per_clip: dict[str, dict] = {}
    for split in SPLITS:
        rows = splits[split]
        with timed(f"{run} {split}: decoding {len(rows)} clips"):
            texts, compute, log = decode(rows)
        with timed(f"{run} {split}: scoring, per clip class and against the references"):
            errors = out / "harness" / "errors" / f"{split}.parquet"
            m, per_clip[split] = score_split(
                rows, texts, compute, log, score, errors=errors, run=run, word_counts=word_counts
            )
            m["vs"] = {}
            for name, ref in (references or {}).items():
                try:
                    m["vs"][name] = pair(rows, ref.get(split) or {}, per_clip[split], keys=keys)
                except ValueError:  # the reference scored none of this split's clips
                    continue
        results[split] = m
        write_hyps(out / "harness" / f"{split}.jsonl", rows, texts, compute)
        (out / f"{split}_metrics.json").write_text(json.dumps(m, indent=1, ensure_ascii=False))
    gold, val = results["gold"], results["val"]
    full_card = {
        **card,
        **(meta or {}),
        "created_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),  # noqa: UP017
        "run_name": run,
        "val_wer": val["wer"],
        "gold_wer": gold["wer"],
        # folded, per 100 reference words; S + D + I = WER
        "val_sid": {k: val[k] for k in ("sub", "del", "ins")},
        "gold_sid": {k: gold[k] for k in ("sub", "del", "ins")},
    }
    (out / "harness" / "model_card.json").write_text(
        json.dumps(full_card, indent=1, ensure_ascii=False)
    )
    (out / "per_clip.json").write_text(json.dumps(per_clip))
    row = {
        "run_name": run,
        **(meta or {}),
        "val_wer": val["wer"],
        "gold_wer": gold["wer"],
        **{split: {k: results[split][k] for k in KEEP} for split in SPLITS},
        "gold_by_class": {k: gold["by_class"].get(k) for k in keys},
    }
    (out / "result.json").write_text(json.dumps(row, indent=1, ensure_ascii=False))
    for split in ("val", "gold"):
        m = results[split]
        print(
            f"{run} {split}: WER {m['wer']:.2f} (S {m['sub']:.2f}  D {m['del']:.2f}  "
            f"I {m['ins']:.2f}), raw {m['raw_wer']:.2f}"
        )
        for name, vs in m["vs"].items():
            d, lo, hi = vs["all"]
            print(f"    minus {name}: {d:+.2f} [{lo:+.2f}, {hi:+.2f}] on {vs['clips']} clips")
        print_breakdown(split, m.get("breakdown"))
    return row


# --- a finished run, scored again under newer rules --------------------------------------------

#: What `evaluate_run` derives from the scores, in `result.json` and the card: everything else in
#: them is the run's recipe, kept as it was when the run is scored again.
_ROW_SCORES = ("run_name", "val_wer", "gold_wer", "gold", "val", "gold_by_class")
_CARD_SCORES = (
    "created_at",
    "rescored_at",
    "run_name",
    "val_wer",
    "gold_wer",
    "val_sid",
    "gold_sid",
)


def stored_decode(out: Path) -> Decode:
    """A `decode` that reads a finished run's transcripts, compute and loop retries from its
    files instead of running a model. A clip the run never transcribed is refused: the rows are
    from another export."""
    out = Path(out)
    texts: dict[str, tuple[str, float]] = {}
    retries: dict[str, tuple[str, str, str]] = {}
    for split in SPLITS:
        hyps = out / "harness" / f"{split}.jsonl"
        if hyps.exists():
            with hyps.open(encoding="utf-8") as fh:
                for j in map(json.loads, fh):
                    texts[j["segment_id"]] = (j["text"], j["compute_s"])
        metrics = out / f"{split}_metrics.json"
        if metrics.exists():
            for r in json.loads(metrics.read_text("utf-8")).get("retried", []):
                retries[r["segment_id"]] = (r["segment_id"], r["first"], r["retry"])

    def decode(rows: Sequence[dict]) -> tuple[list[str], list[float], list[tuple[str, str, str]]]:
        missing = [r["segment_id"] for r in rows if r["segment_id"] not in texts]
        if missing:
            raise ValueError(
                f"{len(missing)} clip(s) this run never transcribed, first {missing[0]!r}: "
                "the rows are from another export"
            )
        ids = [r["segment_id"] for r in rows]
        return (
            [texts[i][0] for i in ids],
            [texts[i][1] for i in ids],
            [retries[i] for i in ids if i in retries],
        )

    return decode


def rescore_run(
    out: Path,
    run: str,
    *,
    splits: Mapping[str, Sequence[dict]],
    score: Any,
    fold_version: str | None = None,
    references: Mapping[str, Mapping[str, Counts]] | None = None,
    keys: Sequence[str] = REPORT_KEYS,
    word_counts: Path | None = None,
) -> dict[str, Any]:
    """Score a finished run again from its stored transcripts, under `score`'s rules (D113).

    The run's files (`evaluate_run`) are rewritten as if it had been scored today: the metrics,
    error rows, per-clip counts, result row and card, with the recipe kept from `result.json`, the
    card's `created_at` kept and `rescored_at` added, and `fold_version` (default:
    `score.fold_version`) set. Nothing is decoded."""
    out = Path(out)
    old_row = json.loads((out / "result.json").read_text("utf-8"))
    old_card = json.loads((out / "harness" / "model_card.json").read_text("utf-8"))
    meta = {k: v for k, v in old_row.items() if k not in _ROW_SCORES}
    card = {k: v for k, v in old_card.items() if k not in _CARD_SCORES and k not in meta}
    card["fold_version"] = fold_version or score.fold_version
    row = evaluate_run(
        out, run, splits=splits, decode=stored_decode(out), score=score, card=card, meta=meta,
        references=references, keys=keys, word_counts=word_counts,
    )  # fmt: skip
    path = out / "harness" / "model_card.json"
    full = json.loads(path.read_text("utf-8"))
    full["created_at"] = old_card.get("created_at", full["created_at"])
    full["rescored_at"] = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")  # noqa: UP017
    path.write_text(json.dumps(full, indent=1, ensure_ascii=False))
    return row


def rescore_benchmark(
    out: Path,
    name: str,
    *,
    score: Any,
    run: str | None = None,
    conditions_dir: Path | None = None,
    word_counts: Path | None = None,
) -> dict[str, Any]:
    """Score a public set again from its stored lines (`run_benchmarks`), under `score`'s rules.

    The lines hold each clip's reference, transcript, group and `by` value, which is all a score
    needs; the hours, speed, retries and limit of the decode are kept from the old summary.
    Rewrites `<name>.json`, `<name>.jsonl` and the set's error rows."""
    out = Path(out)
    folder = out / "benchmarks"
    old = json.loads((folder / f"{name}.json").read_text("utf-8"))
    with (folder / f"{name}.jsonl").open(encoding="utf-8") as fh:
        lines = [json.loads(line) for line in fh]
    by = BENCHMARKS[name].by
    rows = [
        {
            "segment_id": x["id"],
            "episode_id": x["group"],
            "text": x["ref"],
            "start_time": 0.0,
            "end_time": 0.0,
            **({by: x[by]} if by else {}),
        }
        for x in lines
    ]
    summary, new_lines = score_benchmark(
        name,
        rows,
        [x["hyp"] for x in lines],
        score,
        errors=out / "harness" / "errors" / f"{name}.parquet",
        run=run or out.name,
        overlap=_measured(conditions_dir, "overlap", name),
        snr=_measured(conditions_dir, "acoustics", name),
        word_counts=word_counts,
    )
    for key in ("hours", "retried", "x_realtime", "limit"):
        if key in old:
            summary[key] = old[key]
    with (folder / f"{name}.jsonl").open("w", encoding="utf-8") as fh:
        fh.writelines(json.dumps(line, ensure_ascii=False) + "\n" for line in new_lines)
    (folder / f"{name}.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False))
    print(f"{name}: WER {old['wer']:.2f} -> {summary['wer']:.2f}")
    return summary


# --- the public sets -----------------------------------------------------------------------------


@dataclass(frozen=True)
class Benchmark:
    """One public Nepali test set, as it was scored on 2026-09-27 (findings.md).

    `audio`, `text` and `group` are its columns: the clip, the reference, and the unit a paired
    interval resamples (a speaker; FLEURS repeats sentences, so there it is the sentence; nepali_cs
    is resampled by video). `clip_id` names each clip, from one column or several joined by `-`.
    `by` is a column the score is also split by. `base_wer` is base Flex's folded WER on
    2026-09-27: decoding base again should land on it, which is the loader's check."""

    name: str
    what: str
    repo: str
    config: str
    split: str
    audio: str
    text: str
    group: str
    clip_id: tuple[str, ...]
    base_wer: float
    by: str | None = None
    #: "parquet": the hub's converted files. "tar": Common Voice's mirror, a TSV and a tar of MP3s.
    layout: str = "parquet"


BENCHMARKS: dict[str, Benchmark] = {
    b.name: b
    for b in (
        Benchmark(
            name="fleurs",
            what="FLEURS ne_np test: read Wikipedia sentences",
            repo="google/fleurs",
            config="ne_np",
            split="test",
            audio="audio",
            text="raw_transcription",
            group="id",
            clip_id=("path",),
            base_wer=11.10,
        ),
        Benchmark(
            name="slr54",
            what="OpenSLR 54 test: crowd-sourced read speech (the mirror's own split)",
            repo="iamTangsang/OpenSLR54-Nepali-ASR",
            config="default",
            split="test",
            audio="utterance",
            text="transcription",
            group="speaker_id",
            clip_id=("utterance_id",),
            base_wer=8.17,
        ),
        Benchmark(
            name="common_voice",
            what="Common Voice 22 ne-NP test: crowd-sourced read speech",
            repo="fsicoli/common_voice_22_0",
            config="ne-NP",
            split="test",
            audio="path",
            text="sentence",
            group="client_id",
            clip_id=("path",),
            base_wer=8.78,
            layout="tar",
        ),
        Benchmark(
            name="indicvoices",
            what="IndicVoices nepali valid: read, extempore and conversation",
            repo="ai4bharat/IndicVoices",
            config="nepali",
            split="valid",
            audio="audio_filepath",
            text="text",
            group="speaker_id",
            clip_id=(),  # it has none: clips are numbered in file order
            base_wer=12.70,
            by="scenario",
        ),
        Benchmark(
            name="nepali_cs",
            what="nepali-cs-asr test: Nepali-English code-switched lectures",
            repo="saileshbro/nepali-cs-asr",
            config="default",
            split="test",
            audio="audio",
            text="transcription",
            group="video_id",
            clip_id=("video_id", "start_ms"),
            base_wer=10.98,
            by="speech_type",
        ),
    )
}


def plain_tokens(text: str | None) -> list[str]:
    """The normalisation published Nepali results use: NFC, punctuation and symbols dropped,
    Latin lowercased, and nothing else. No fold: `टिम` and `team` are different words here."""
    text = unicodedata.normalize("NFC", text or "")
    kept = "".join(c if unicodedata.category(c)[0] in "LMN" else " " for c in text)
    return kept.lower().split()


def edit_distance(ref: Sequence[str], hyp: Sequence[str]) -> int:
    """Word-level Levenshtein distance: the errors of a plain WER."""
    row = list(range(len(hyp) + 1))
    for i, r in enumerate(ref, 1):
        prev, row[0] = row[0], i
        for j, h in enumerate(hyp, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (r != h))
    return row[-1]


def plain_wer(refs: Sequence[str], hyps: Sequence[str]) -> float:
    """Pooled plain WER in percent: errors over reference words, across all clips."""
    errors = words = 0
    for ref, hyp in zip(refs, hyps, strict=True):
        r = plain_tokens(ref)
        errors += edit_distance(r, plain_tokens(hyp))
        words += len(r)
    return 100 * errors / max(words, 1)


def plain_cer(refs: Sequence[str], hyps: Sequence[str]) -> float:
    """Pooled plain CER in percent, over the characters of the plain text with its spaces.

    Published Nepali results report it beside WER. A word split differently costs a plain WER
    two words but a CER one space, so the two together tell hearing from spacing."""
    errors = chars = 0
    for ref, hyp in zip(refs, hyps, strict=True):
        r = " ".join(plain_tokens(ref))
        errors += edit_distance(r, " ".join(plain_tokens(hyp)))
        chars += len(r)
    return 100 * errors / max(chars, 1)


def _audio(data: bytes) -> Any:
    """Any encoded clip as 16 kHz mono int16: libsndfile first, ffmpeg for what it cannot read."""
    import numpy as np
    import soundfile as sf

    try:
        x, sr = sf.read(io.BytesIO(data), dtype="float32", always_2d=True)
        x = x.mean(axis=1)
        if sr != SR:
            from math import gcd

            from scipy.signal import resample_poly

            g = gcd(SR, sr)
            x = resample_poly(x, SR // g, sr // g)
        return np.round(np.clip(x, -1.0, 1.0) * 32767).astype(np.int16)
    except Exception:
        cmd = [
            "ffmpeg",
            "-v",
            "error",
            "-i",
            "pipe:0",
            "-f",
            "s16le",
            "-ac",
            "1",
            "-ar",
            str(SR),
            "pipe:1",
        ]
        raw = subprocess.run(cmd, input=data, capture_output=True, check=True).stdout
        return np.frombuffer(raw, dtype=np.int16).copy()


def benchmark_row(bench: Benchmark, record: Mapping[str, Any], index: int, audio: Any) -> dict:
    """One clip as the decoders and the scorer read a row: it carries its own audio, its group
    stands where an episode does, and `segment_id` is the id the 2026-09-27 files used."""
    if bench.clip_id:
        clip_id = "-".join(str(record[c]) for c in bench.clip_id).rsplit("/", 1)[-1]
    else:
        clip_id = f"iv-{index:05d}"
    row = {
        "segment_id": clip_id,
        "episode_id": str(record[bench.group]),
        "text": record[bench.text] or "",
        "audio": audio,
        "start_time": 0.0,
        "end_time": len(audio) / SR,
    }
    if bench.by:
        row[bench.by] = record[bench.by]
    return row


def _parquet_records(bench: Benchmark, work: Path, token: str) -> Iterator[tuple[dict, bytes]]:
    """(record, encoded audio) from the hub's parquet conversion of the split, a file at a time."""
    import pyarrow.parquet as pq
    import requests

    headers = {"Authorization": f"Bearer {token}"}
    listing = (
        f"https://huggingface.co/api/datasets/{bench.repo}/parquet/{bench.config}/{bench.split}"
    )
    urls = requests.get(listing, headers=headers, timeout=60)
    urls.raise_for_status()
    columns = sorted(
        {bench.audio, bench.text, bench.group, *bench.clip_id, *([bench.by] if bench.by else [])}
    )
    for k, url in enumerate(urls.json()):
        path = work / f"{bench.name}-{k:04d}.parquet"
        if not path.exists():
            with requests.get(url, headers=headers, stream=True, timeout=600) as r:
                r.raise_for_status()
                with path.with_suffix(".part").open("wb") as fh:
                    for chunk in r.iter_content(1 << 22):
                        fh.write(chunk)
            path.with_suffix(".part").rename(path)
        table = pq.ParquetFile(path)
        missing = set(columns) - set(table.schema_arrow.names)
        if missing:
            raise KeyError(
                f"{bench.repo}: no column {sorted(missing)}; it has {table.schema_arrow.names}"
            )
        for batch in table.iter_batches(batch_size=64, columns=columns):
            for record in batch.to_pylist():
                yield record, record[bench.audio]["bytes"]


def _tar_records(bench: Benchmark, token: str) -> Iterator[tuple[dict, bytes]]:
    """(record, encoded audio) from the Common Voice mirror: `transcript/<locale>/<split>.tsv`
    and the MP3s in `audio/<locale>/<split>/<locale>_<split>_<n>.tar`, in the TSV's order."""
    import csv

    from huggingface_hub import hf_hub_download, list_repo_files

    def fetch(name: str) -> str:
        return hf_hub_download(bench.repo, name, repo_type="dataset", token=token)

    tsv = fetch(f"transcript/{bench.config}/{bench.split}.tsv")
    with open(tsv, encoding="utf-8", newline="") as fh:
        records = list(csv.DictReader(fh, delimiter="\t", quoting=csv.QUOTE_NONE))
    prefix = f"audio/{bench.config}/{bench.split}/"
    clips: dict[str, bytes] = {}
    for name in sorted(list_repo_files(bench.repo, repo_type="dataset", token=token)):
        if name.startswith(prefix) and name.endswith(".tar"):
            with tarfile.open(fetch(name)) as tf:
                for member in tf:
                    if member.isfile():
                        clips[member.name.rsplit("/", 1)[-1]] = tf.extractfile(member).read()
    for record in records:
        yield record, clips[record[bench.audio]]


def iter_benchmark(name: str, work: Path, token: str) -> Iterator[dict]:
    """A public set's clips one at a time, in file order, each a row with its audio (16 kHz mono
    int16): for a pass that must not hold the whole set in RAM."""
    bench = BENCHMARKS[name]
    work = Path(work)
    work.mkdir(parents=True, exist_ok=True)
    records = (
        _tar_records(bench, token)
        if bench.layout == "tar"
        else _parquet_records(bench, work, token)
    )
    for index, (record, data) in enumerate(records):
        yield benchmark_row(bench, record, index, _audio(data))


def load_benchmark(name: str, work: Path, token: str, limit: int | None = None) -> list[dict]:
    """A public set's clips as rows with their audio in RAM (16 kHz mono int16), in file order.

    `limit` keeps the first clips only, for a smoke run. Nothing here is cached beyond the
    downloaded files: a set is loaded, decoded, scored and dropped."""
    rows = []
    for row in iter_benchmark(name, work, token):
        if limit is not None and len(rows) >= limit:
            break
        rows.append(row)
    return rows


def score_benchmark(
    name: str,
    rows: Sequence[dict],
    texts: Sequence[str],
    score: Any,
    *,
    errors: Path | None = None,
    run: str = "",
    overlap: Mapping[str, float | None] | None = None,
    snr: Mapping[str, float | None] | None = None,
    word_counts: Path | None = None,
) -> tuple[dict[str, Any], list[dict]]:
    """A public set's summary and its per-clip lines.

    Summary: folded WER with S/D/I, raw WER and CER from the harness's scorer, plain WER and CER,
    and the score per value of the set's `by` column. A line is `{id, group, ref, hyp, errors,
    words}`: the 2026-09-27 files' fields plus the counts a paired comparison needs. With
    `errors`, the set's error rows are written there, crosstalk from `overlap` and SNR from `snr`
    (clips missing from either are unmeasured on it), and its `breakdown` added to the summary,
    split by word rarity with `word_counts`."""
    bench = BENCHMARKS[name]
    refs = [r["text"] for r in rows]
    clips = score.per_clip(refs, texts)
    summary = score.summarize(clips)
    summary["plain_wer"] = plain_wer(refs, texts)
    summary["plain_cer"] = plain_cer(refs, texts)
    summary["hours"] = sum(duration(r) for r in rows) / 3600
    if bench.by:
        values: dict[str, list[int]] = {}
        for i, r in enumerate(rows):
            values.setdefault(str(r[bench.by]), []).append(i)
        summary["by"] = {
            v: score.summarize([clips[i] for i in idx]) for v, idx in sorted(values.items())
        }
    if errors is not None:
        summary["breakdown"] = write_errors(
            errors,
            run,
            rows,
            texts,
            clips,
            overlap=overlap or {},
            snr=snr or {},
            by=bench.by,
            word_counts=word_counts,
        )
    lines = []
    for r, hyp, c in zip(rows, texts, clips, strict=True):
        line = {
            "id": r["segment_id"],
            "group": r["episode_id"],
            "ref": r["text"],
            "hyp": hyp,
            "errors": c["errors"],
            "words": c["words"],
        }
        if bench.by:
            line[bench.by] = r[bench.by]
        lines.append(line)
    return summary, lines


def pair_benchmark(a: Sequence[dict], b: Sequence[dict], n: int = 2000) -> dict[str, Any]:
    """WER(b) - WER(a) on one public set from two models' per-clip lines, resampling the set's
    group; `{clips, all: [difference, low, high]}`."""
    rows = [{"segment_id": x["id"], "episode_id": x["group"]} for x in b]
    return pair(
        rows,
        {x["id"]: [x["errors"], x["words"]] for x in a},
        {x["id"]: [x["errors"], x["words"]] for x in b},
        keys=(),
        n=n,
    )


def _measured(folder: Path | None, kind: str, name: str) -> dict[str, float | None] | None:
    """`folder/<kind>/<name>.parquet` as values by clip id, or None when absent -- or when the
    harness copy predates the reader (SNR came with mine-v2)."""
    path = Path(folder) / kind / f"{name}.parquet" if folder is not None else None
    if path is None or not path.exists() or (kit := _miner()) is None:
        return None
    read = getattr(kit[0], "read_overlap" if kind == "overlap" else "read_snr", None)
    return read(path) if read is not None else None


def run_benchmarks(
    out: Path,
    *,
    decode: Decode,
    score: Any,
    work: Path,
    token: str,
    names: Sequence[str] = tuple(BENCHMARKS),
    limit: int | None = None,
    done: Collection[str] = (),
    run: str | None = None,
    conditions_dir: Path | None = None,
    word_counts: Path | None = None,
) -> dict[str, dict]:
    """Decode and score each public set with the weights `decode` reads, one set at a time.

    Writes `out/benchmarks/<name>.jsonl` (per clip) and `<name>.json` (summary), and the set's
    error rows to `out/harness/errors/<name>.parquet`, as each set finishes, and skips a set named
    in `done`, so a lost runtime costs one set. `run` names the rows (default: `out`'s folder);
    `conditions_dir` holds the sets' measured crosstalk and acoustics (`fetch_conditions`), and
    `word_counts` the training corpus's (`word_counts_path`)."""
    run = run or Path(out).name
    folder = Path(out) / "benchmarks"
    folder.mkdir(parents=True, exist_ok=True)
    summaries = {}
    for name in names:
        if name in done:
            print(f"{name}: already scored, skipped")
            continue
        with timed(f"{name}: loading"):
            rows = load_benchmark(name, work, token, limit)
        with timed(f"{name}: decoding {len(rows)} clips"):
            texts, compute, log = decode(rows)
        with timed(f"{name}: scoring"):
            summary, lines = score_benchmark(
                name,
                rows,
                texts,
                score,
                errors=Path(out) / "harness" / "errors" / f"{name}.parquet",
                run=run,
                overlap=_measured(conditions_dir, "overlap", name),
                snr=_measured(conditions_dir, "acoustics", name),
                word_counts=word_counts,
            )
        summary["retried"] = len(log)
        summary["x_realtime"] = sum(duration(r) for r in rows) / max(sum(compute), 1e-9)
        summary["limit"] = limit
        with (folder / f"{name}.jsonl").open("w", encoding="utf-8") as fh:
            fh.writelines(json.dumps(line, ensure_ascii=False) + "\n" for line in lines)
        (folder / f"{name}.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False))
        print(
            f"{name}: WER {summary['wer']:.2f} (S {summary['sub']:.2f}  D {summary['del']:.2f}  "
            f"I {summary['ins']:.2f}), plain {summary['plain_wer']:.2f} / CER "
            f"{summary['plain_cer']:.2f}, {len(rows)} clips, {summary['x_realtime']:.0f}x realtime"
        )
        print_breakdown(name, summary.get("breakdown"))
        summaries[name] = summary
        del rows
    return summaries

In [ ]:
%%writefile /content/ft/sweep.py
"""Which run's weights are kept, and how two runs are compared (D96, D105).

A notebook trains several runs on the same export: the crosstalk sweep's `(XTALK_P, seed)` points
(D96), the augmentation ablation's named recipes, or the weight blends of one fine-tune (D105).
Each winner is chosen by val WER alone, under a rule fixed before any result was read. For
training runs: a recipe has to beat the baseline by more than the seed noise, measured as the gap
between the baseline's two seeds, or the baseline is kept. For named recipes that gap is never
taken as less than `MIN_NOISE` (D109). Gold never takes part in a choice, so every gold number
stays a held-out score.

Runs are compared on gold with a paired bootstrap that resamples whole episodes: clips of one
episode share a room and voices, and resampling them one by one would pretend to more independent
evidence than gold holds (findings.md, *How big gold has to be*). Pure numpy; importable without
torch.
"""

from __future__ import annotations

from collections.abc import Sequence

import numpy as np


def run_name(prefix: str, p: float, seed: int) -> str:
    """`<prefix>-p30-s0` for XTALK_P = 0.3 and seed 0: the run's folder and Models page id."""
    return f"{prefix}-p{round(p * 100):02d}-s{seed}"


def _winner(
    base: list[dict], other: list[dict], base_name: str, label, floor: float = 0.0
) -> tuple[dict, str]:
    """The rule every chooser shares. `base` and `other` are sorted by val WER, best first.

    The bar is the gap between the baseline's two best seeds, or `floor` when that is larger
    (D109). Both are rounded to 1e-6 so a margin equal to the bar is a tie, not a float win."""
    if not base or not other:
        best = (base or other)[0]
        return best, "only one kind of run, so the lowest val WER"
    if len(base) >= 2:
        noise = base[1]["val_wer"] - base[0]["val_wer"]
        seeds = f"{base_name} seeds {base[0]['seed']} and {base[1]['seed']}"
        noise_note = f"seed noise {noise:.2f} ({seeds})"
    else:
        noise = 0.0
        noise_note = f"seed noise unmeasured (one {base_name} seed)"
        if not floor:
            noise_note += ", so the rule is strict"
    if floor > noise:
        noise, noise_note = floor, f"the floor {floor:.2f}, as {noise_note} is below it"
    margin = base[0]["val_wer"] - other[0]["val_wer"]
    if round(margin, 6) > round(noise, 6):
        return other[
            0
        ], f"{label(other[0])} beats {base_name} on val by {margin:.2f} > {noise_note}"
    return base[0], f"best augmented margin {margin:.2f} is within {noise_note}; {base_name} kept"


def choose_winner(rows: Sequence[dict]) -> tuple[dict, str]:
    """The run whose weights are kept, and why, from rows with `xtalk_p`, `seed` and `val_wer`.

    p = 0 is its better seed. The noise is the gap between its two best seeds; with one p = 0 seed
    it is unmeasured and the rule is strict. The best augmented run wins only if it beats p = 0 by
    more than the noise; a tie goes to p = 0, the simpler recipe."""
    if not rows:
        raise ValueError("no runs to choose from")
    base = sorted((r for r in rows if r["xtalk_p"] == 0), key=lambda r: r["val_wer"])
    aug = sorted((r for r in rows if r["xtalk_p"] != 0), key=lambda r: r["val_wer"])
    return _winner(base, aug, "p=0", lambda r: f"p={r['xtalk_p']}")


#: The recipe every augmentation is measured against (D105): today's training, nothing added.
BASELINE = "vanilla"

#: The least a named recipe has to beat the baseline by on val, in WER points (D109). Two seeds
#: can agree by luck: 03a's differed by 0.02 on val and 0.08 on gold. Below this the seed gap is
#: not trusted as the noise.
MIN_NOISE = 0.15


def choose_recipe(
    rows: Sequence[dict], baseline: str = BASELINE, floor: float = MIN_NOISE
) -> tuple[dict, str]:
    """`choose_winner` for named recipes (D105): rows carry `recipe`, `seed` and `val_wer`.

    The baseline is its better seed, and the noise is the gap between its two best seeds, or
    `floor` when that is larger (D109). Another recipe wins only if it beats the baseline on val
    by more than that; otherwise the baseline is kept. Gold and the public sets never take part."""
    if not rows:
        raise ValueError("no runs to choose from")
    base = sorted((r for r in rows if r["recipe"] == baseline), key=lambda r: r["val_wer"])
    other = sorted((r for r in rows if r["recipe"] != baseline), key=lambda r: r["val_wer"])
    return _winner(base, other, baseline, lambda r: r["recipe"], floor)


def beats_baseline(
    row: dict, baseline_rows: Sequence[dict], baseline: str = BASELINE, floor: float = MIN_NOISE
) -> tuple[bool, str]:
    """Whether one run clears the rule on its own, and why: what decides, as each ablation run
    finishes, if its stage goes into the combined recipe and its weights are kept."""
    base = [r for r in baseline_rows if r["recipe"] == baseline]
    if not base:
        raise ValueError(f"no {baseline} run to measure against")
    winner, why = choose_recipe([*base, row], baseline, floor)
    return winner is row, why


def choose_blend(rows: Sequence[dict], tolerance: float = 0.3) -> tuple[dict, str]:
    """The weight blend that is kept, and why, from rows with `alpha` and `val_wer`.

    A blend is `(1 - alpha) * base + alpha * fine-tuned` (WiSE-FT), so alpha = 1 is the fine-tuned
    model and must be among the rows. The rule, fixed on 2026-09-27 before any blend was scored:
    the blend closest to base whose val WER is within `tolerance` points of the fine-tuned
    model's. Base itself (alpha = 0) is never a candidate, and if no blend qualifies the
    fine-tuned model is kept. Gold and the public sets never take part."""
    tuned = [r for r in rows if r["alpha"] == 1]
    if len(tuned) != 1:
        raise ValueError("the rule needs exactly one alpha = 1 row, the fine-tuned model")
    limit = tuned[0]["val_wer"] + tolerance
    ok = sorted(
        (r for r in rows if 0 < r["alpha"] < 1 and r["val_wer"] <= limit), key=lambda r: r["alpha"]
    )
    if not ok:
        return tuned[0], f"no blend is within {tolerance:g} of the fine-tuned val WER; it is kept"
    return ok[0], (
        f"alpha={ok[0]['alpha']:g} is the blend closest to base with val WER "
        f"{ok[0]['val_wer']:.2f} <= {tuned[0]['val_wer']:.2f} + {tolerance:g}"
    )


def paired_bootstrap(
    a: Sequence[dict],
    b: Sequence[dict],
    episodes: Sequence[str],
    *,
    n: int = 2000,
    seed: int = 0,
    level: float = 0.95,
) -> tuple[float, float, float]:
    """WER(b) - WER(a) in points, with a `level` (95%) interval from resampling episodes.

    `a` and `b` are per-clip `{"errors", "words"}` for the same clips in the same order, and
    `episodes` names each clip's episode. WER is pooled (sum of errors over sum of words)."""
    if not (len(a) == len(b) == len(episodes)):
        raise ValueError("a, b and episodes must describe the same clips")
    ids = sorted(set(episodes))
    index = {e: i for i, e in enumerate(ids)}
    ep = np.array([index[e] for e in episodes])
    per = np.zeros((len(ids), 3))  # errors of a, errors of b, reference words
    clips = [[x["errors"], y["errors"], x["words"]] for x, y in zip(a, b, strict=True)]
    np.add.at(per, ep, np.array(clips))

    def diff(t: np.ndarray) -> float:
        return float(100 * (t[1] - t[0]) / max(t[2], 1))

    point = diff(per.sum(axis=0))
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, len(ids), size=(n, len(ids)))
    totals = per[draws].sum(axis=1)  # (n, 3)
    boot = 100 * (totals[:, 1] - totals[:, 0]) / np.maximum(totals[:, 2], 1)
    tail = 100 * (1 - level) / 2
    lo, hi = np.percentile(boot, [tail, 100 - tail])
    return point, float(lo), float(hi)

In [ ]:
%%writefile /content/ft/distill.py
"""Pure helpers for the distillation notebook (roadmap §B).

No torch and no NeMo, so the rules that decide what shapes the students' tokenizer and what their
scores are paired against are tested from backend/tests. Keep it importable on Python 3.10+.
"""

from __future__ import annotations

import re
import unicodedata
from collections.abc import Callable, Mapping, Sequence
from typing import Any


def smoke_source(
    smoke: bool, smoke_path: str, real_path: str, exists: Callable[[str], bool]
) -> str:
    """The output a notebook reads from its predecessor: the smoke one if it exists, else the real.

    A smoke run only reads what came before it and writes under its own smoke names, so reading a
    real output is safe; it is what lets a smoke run follow a predecessor that ran for real
    without one. A real run never reads a smoke output."""
    if smoke and exists(smoke_path):
        return smoke_path
    return real_path


def smoke_name(smoke: bool, stem: str) -> str:
    """The name a notebook writes `stem` under: `<stem>-smoke` in a smoke run. A smoke run may
    read a real folder (`smoke_source`), so what it writes there must never take a real name."""
    return f"{stem}-smoke" if smoke else stem


def tokenizer_texts(splits: Mapping[str, Sequence[dict]]) -> list[str]:
    """The labels the shared student tokenizer is trained on: train only.

    Val and gold text never shapes the vocabulary, so a word only they contain is scored as the
    student would meet it on new audio. Raises if a val or gold clip is in train."""
    held = {r["segment_id"] for name in ("val", "gold") for r in splits.get(name, ())}
    leaked = sorted(held & {r["segment_id"] for r in splits["train"]})
    if leaked:
        raise ValueError(f"held-out clips in train: {leaked[:5]}")
    return [r["text"] for r in splits["train"]]


def reference_texts(rows: Sequence[dict], by_id: Mapping[str, str]) -> list[str]:
    """Another system's transcripts in `rows`' order, for pairing clip by clip. Raises, naming
    them, if any row has none: a pairing over a subset would not be the same clips."""
    missing = [r["segment_id"] for r in rows if r["segment_id"] not in by_id]
    if missing:
        raise ValueError(f"{len(missing)} clip(s) have no reference transcript: {missing[:5]}")
    return [by_id[r["segment_id"]] for r in rows]


def by_class(
    rows: Sequence[dict],
    clips: Sequence[Any],
    summarize: Callable[[list[Any]], Any],
    keys: Sequence[str] | None = None,
) -> dict[str, dict[str, Any]]:
    """`summarize` per value of each clip class (D87), as {key: {value: summary}}.

    `clips` holds one per-clip count per row (`score.per_clip`), aligned once by the caller: a
    group's score is its clips' counts added up, so no clip is aligned again per key. `keys`
    defaults to every class key any row carries. A clip without a key (or with no classes) is left
    out of that key's groups, not counted under a made-up value."""
    if len(rows) != len(clips):
        raise ValueError("rows and clips must describe the same clips")
    if keys is None:
        keys = sorted({k for r in rows for k in (r.get("classes") or {})})
    out: dict[str, dict[str, Any]] = {}
    for key in keys:
        groups: dict[str, list[int]] = {}
        for i, r in enumerate(rows):
            value = (r.get("classes") or {}).get(key)
            if value is not None:
                groups.setdefault(str(value), []).append(i)
        if groups:
            out[key] = {v: summarize([clips[i] for i in idx]) for v, idx in sorted(groups.items())}
    return out


def filter_pseudo(
    rows: Sequence[dict],
    *,
    tokens_per_s: tuple[float, float],
    drop_fraction: float,
    max_overlap_share: float | None = None,
) -> tuple[list[dict], dict[str, Any]]:
    """Roadmap §B step 3: keep the teacher's labels worth training on, cheapest rule first.

    Each row carries ``segment_id``, ``text``, ``n_tokens`` (teacher tokens), ``duration`` (s),
    ``mean_logprob`` and ``looped`` (the greedy output loops, ``ftkit.is_loop``), and, once
    PreDistill has measured it, ``overlap_share``. Dropped, in order: a clip overlapped for more
    than ``max_overlap_share`` of its length (crosstalk is where the teacher is weakest; ``None``
    applies no such rule, and a clip never measured is kept and counted); a clip whose output
    loops; an empty transcript; a rate outside ``tokens_per_s`` (the range train's labels span, in
    the teacher's tokens); then the least confident ``drop_fraction`` of what is left, by mean
    log-prob. Returns the kept rows, in input order, and a report of what each rule dropped.
    """
    if not 0.0 <= drop_fraction < 1.0:
        raise ValueError(f"drop_fraction must be in [0, 1), not {drop_fraction}")
    if max_overlap_share is not None and not 0.0 <= max_overlap_share <= 1.0:
        raise ValueError(f"max_overlap_share must be in [0, 1], not {max_overlap_share}")
    lo, hi = tokens_per_s
    dropped = {"overlap": 0, "loop": 0, "empty": 0, "rate": 0, "confidence": 0}
    unmeasured = sum(r.get("overlap_share") is None for r in rows)
    left = []
    for r in rows:
        share = r.get("overlap_share")
        if max_overlap_share is not None and share is not None and share > max_overlap_share:
            dropped["overlap"] += 1
        elif r["looped"]:
            dropped["loop"] += 1
        elif not r["text"].strip():
            dropped["empty"] += 1
        elif not lo <= r["n_tokens"] / r["duration"] <= hi:
            dropped["rate"] += 1
        else:
            left.append(r)
    cut = None
    n_drop = int(len(left) * drop_fraction)
    if n_drop:
        least = sorted(left, key=lambda r: r["mean_logprob"])[:n_drop]
        cut = max(r["mean_logprob"] for r in least)
        gone = {r["segment_id"] for r in least}
        left = [r for r in left if r["segment_id"] not in gone]
        dropped["confidence"] = n_drop
    report = {
        "total": len(rows),
        "kept": len(left),
        "kept_hours": round(sum(r["duration"] for r in left) / 3600, 3),
        "dropped": dropped,
        "logprob_cut": cut,
        "tokens_per_s": [lo, hi],
        "drop_fraction": drop_fraction,
        "max_overlap_share": max_overlap_share,
        "overlap_unmeasured": unmeasured,
    }
    return left, report


def latin_share(text: str) -> float:
    """The share of a transcript's words written in Latin script: a code-mixing proxy that needs
    only the text, for pseudo-labels, which have no CMI class."""
    words = [w for w in text.split() if any(c.isalpha() for c in w)]
    if not words:
        return 0.0
    return sum(any("a" <= c.lower() <= "z" for c in w) for w in words) / len(words)


def mixing_bucket(share: float) -> str:
    """``latin_share`` in the corpus's CMI buckets (D87): 0, <15, 15-30, 30+ percent."""
    if share == 0:
        return "0"
    if share < 0.15:
        return "<15"
    return "15-30" if share < 0.30 else "30+"


# --- PreDistill: the owner's zip of recordings (D101) ------------------------------------------

_NUMBER = re.compile(r"_\d+$")


def _path_safe(text: str) -> str:
    """Letters and digits of any script, combining marks (Devanagari's vowel signs, which regex's
    word class misses), ``.``, ``-`` and ``_`` kept; every other run of characters becomes one
    ``_``."""
    kept = "".join(
        c if c.isalnum() or c in "._-" or unicodedata.category(c).startswith("M") else "\0"
        for c in text
    )
    return re.sub("\0+", "_", kept).strip("_")


def source_from_filename(name: str) -> tuple[str, str]:
    """A recording named ``<channel_name>_<NN>.mp3`` as ``(source_id, channel)``.

    The source id is the file's stem with every run of characters unsafe in a path turned into
    one ``_`` (letters of any script, digits, ``.``, ``-`` and ``_`` are kept); the channel is the
    id less its trailing ``_<number>``.
    """
    stem = name.replace("\\", "/").rsplit("/", 1)[-1].rsplit(".", 1)[0]
    source_id = _path_safe(stem)
    return source_id, _NUMBER.sub("", source_id) or source_id


def channel_blocked(channel: str, blocked: Sequence[str]) -> bool:
    """Whether a channel's name contains a blocked name, ignoring case and ``_`` against spaces."""
    name = channel.replace("_", " ").casefold()
    return any(b.strip().casefold() in name for b in blocked if b.strip())


def slice_rows(source_id: str, channel: str, slices: Sequence[tuple[float, float]]) -> list[dict]:
    """The manifest rows of one recording's VAD slices, named and timed as the labelled export's
    rows are, so ``ftkit.AudioStore`` cuts them from the whole recording."""
    return [
        {
            "segment_id": f"{source_id}_{i:05d}",
            "episode_id": source_id,
            "source_id": source_id,
            "channel": channel,
            "start_time": round(start, 3),
            "end_time": round(end, 3),
            "duration": round(end - start, 3),
        }
        for i, (start, end) in enumerate(slices)
    ]


# --- overlapped speech on the unlabelled corpus (D105) -------------------------------------------
#
# The harness measures overlap per episode (app/services/overlap.py) and classes a clip by the
# share of it that is overlapped (app/services/clip_classes.py). The second module imports
# SQLAlchemy and the models, so it cannot be fetched into Colab; its two pure functions are
# repeated here, and a test keeps them equal to the harness's.


def spans_within(
    spans: Sequence[Sequence[float]], start: float, end: float
) -> list[tuple[float, float]]:
    """A recording's overlap spans cut to one clip, clip-relative, as the labelled export's
    ``overlap_spans`` are. Empty when the clip holds none."""
    out: list[tuple[float, float]] = []
    for lo, hi in spans:
        a, b = max(lo, start), min(hi, end)
        if b > a:
            out.append((round(a - start, 3), round(b - start, 3)))
    return out


def overlap_share(spans: Sequence[Sequence[float]] | None, duration: float) -> float | None:
    """The fraction of a clip spent in crosstalk; ``None`` when it was never measured."""
    if spans is None:
        return None
    if duration <= 0:
        return 0.0
    return min(1.0, sum(max(0.0, end - start) for start, end in spans) / duration)


def overlap_bucket(share: float | None) -> str:
    """The overlap bucket of a clip's overlap share, the corpus's clip class (D87)."""
    if share is None:
        return "unmeasured"
    if share <= 0:
        return "none"
    if share < 0.05:
        return "0-5%"
    if share <= 0.15:
        return "5-15%"
    return ">15%"


def with_overlap(rows: Sequence[dict], spans: Sequence[Sequence[float]] | None) -> list[dict]:
    """One recording's clip rows with ``overlap_spans`` (clip-relative) and ``overlap_share``,
    from the recording's overlap spans. ``None`` spans (no detector) leave both ``None``: never
    measured, which is not the same as clean."""
    out = []
    for r in rows:
        mine = None if spans is None else spans_within(spans, r["start_time"], r["end_time"])
        share = overlap_share(mine, r["end_time"] - r["start_time"])
        out.append(
            {
                **r,
                "overlap_spans": None if mine is None else [list(s) for s in mine],
                "overlap_share": None if share is None else round(share, 4),
            }
        )
    return out


def overlap_loss(rows: Sequence[dict], thresholds: Sequence[float]) -> dict[str, dict[str, Any]]:
    """What a ``max_overlap_share`` would cost, per channel and for ``all``: the hours of speech,
    the hours never measured, and the hours dropped at each threshold (a clip is dropped when its
    share is *above* the threshold, as ``filter_pseudo`` drops it). The table the owner reads
    before choosing the threshold."""
    out: dict[str, dict[str, Any]] = {}
    for r in rows:
        hours = (r["end_time"] - r["start_time"]) / 3600
        share = r.get("overlap_share")
        for name in (r["channel"], "all"):
            c = out.setdefault(
                name,
                {
                    "hours": 0.0,
                    "unmeasured_hours": 0.0,
                    "dropped_hours": dict.fromkeys(thresholds, 0.0),
                },
            )
            c["hours"] += hours
            if share is None:
                c["unmeasured_hours"] += hours
                continue
            for threshold in thresholds:
                if share > threshold:
                    c["dropped_hours"][threshold] += hours
    return dict(sorted(out.items(), key=lambda kv: (kv[0] == "all", kv[0])))


# --- stage 2 of a student: pseudo-labels and human labels in one mixture (D106) ------------------


def mixture_weights(
    groups: Sequence[str | None], hours: Sequence[float], human_share: float
) -> list[float]:
    """Each clip's chance of being drawn, summing to 1.

    ``groups[i]`` is clip i's channel for a pseudo-labelled clip and ``None`` for a human-labelled
    one; ``hours[i]`` is its length. The human labels take ``human_share`` of the draws. The rest
    is split between channels in proportion to the *square root* of their hours (roadmap §B,
    2026-09-26: no channel is capped, and a 22 h channel weighs about 2x a 5 h one, not 4.4x).
    Inside a channel, and inside the human labels, a clip is drawn in proportion to its length.
    With no pseudo-labelled clip, or no human one, the other kind takes every draw.
    """
    if len(groups) != len(hours):
        raise ValueError("groups and hours must describe the same clips")
    if not 0.0 < human_share < 1.0:
        raise ValueError(f"human_share must be in (0, 1), not {human_share}")
    total: dict[str | None, float] = {}
    for g, h in zip(groups, hours, strict=True):
        if h <= 0:
            raise ValueError("every clip needs a positive length")
        total[g] = total.get(g, 0.0) + h
    channels = {g: h for g, h in total.items() if g is not None}
    roots = sum(h**0.5 for h in channels.values())
    share: dict[str | None, float] = {}
    if None in total:
        share[None] = human_share if channels else 1.0
    for g, h in channels.items():
        share[g] = (1.0 - human_share if None in total else 1.0) * h**0.5 / roots
    return [share[g] * h / total[g] for g, h in zip(groups, hours, strict=True)]


def draw_epoch(weights: Sequence[float], n: int, seed: int) -> list[int]:
    """The ``n`` clips of one epoch, as indices drawn with replacement by ``weights``; repeatable
    from ``seed``, so a resumed run trains the epochs it would have."""
    import random

    return random.Random(seed).choices(range(len(weights)), weights=list(weights), k=n)

In [ ]:
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

import distill
import evalkit

TOKEN = os.environ["HF_TOKEN"]
api = HfApi(token=TOKEN)


def fetch_json(repo, remote):
    # A JSON file of a model repo, or None when it is not there.
    if not api.file_exists(repo, remote):
        return None
    return json.loads(Path(hf_hub_download(repo, remote, token=TOKEN)).read_text("utf-8"))


teacher = fetch_json(FLEX_REPO, distill.smoke_source(SMOKE, "teacher-smoke.json", "teacher.json",
                                                     lambda path: api.file_exists(FLEX_REPO, path)))
if teacher is None:
    raise RuntimeError(f"{FLEX_REPO} has no teacher.json: run 03e_Flex_Ship.ipynb first")
FLEX_PREFIX, TEACHER = teacher["run"].rsplit("/", 1)

# The labels, for each clip's episode and classes; no audio.
root = Path(snapshot_download(DATASET_REPO, repo_type="dataset", token=TOKEN,
                              allow_patterns=["training/training.jsonl", "training/manifest.json", "gold/gold.jsonl"]))
export = json.loads((root / "training" / "manifest.json").read_text())
evalkit.check_export(export, DATASET_EXPORT)
training = [json.loads(line) for line in (root / "training" / "training.jsonl").open(encoding="utf-8")]
rows = {"gold": [json.loads(line) for line in (root / "gold" / "gold.jsonl").open(encoding="utf-8")],
        "val": [r for r in training if r["split"] == "val"]}


def runs_of(repo, prefix):
    # The run folders under `prefix` that hold a result.
    return sorted({f.split("/")[1] for f in api.list_repo_files(repo)
                   if f.startswith(prefix + "/") and f.endswith("/result.json") and f.count("/") == 2})


models, left_out = [], []
for repo, prefix in ((FLEX_REPO, FLEX_PREFIX), (STUDENTS_REPO, STUDENTS_PREFIX)):
    for run in runs_of(repo, prefix):
        card = fetch_json(repo, f"{prefix}/{run}/harness/model_card.json") or {}
        if card.get("dataset_export") != export["exported_at"]:
            left_out.append((run, card.get("dataset_export")))
            continue
        models.append({"repo": repo, "prefix": prefix, "run": run, "card": card,
                       "row": fetch_json(repo, f"{prefix}/{run}/result.json"),
                       "counts": fetch_json(repo, f"{prefix}/{run}/per_clip.json") or {}})
by_run = {m["run"]: m for m in models}
if TEACHER not in by_run:
    raise RuntimeError(f"the teacher, {TEACHER}, has no result on this export under {FLEX_REPO}/{FLEX_PREFIX}")


def order(m):
    # base, the Flex runs, the teacher last among them, then the students by name and stage
    row = m["row"]
    stage = {"human": 0, "distill": 1, "distill-aug": 2}.get(row.get("stage"), 0)
    return (m["repo"] != FLEX_REPO, row.get("student") or "", stage,
            m["run"] != "base", m["run"] == TEACHER, m["run"])


models.sort(key=order)
print(f"export {export['exported_at'][:10]} | teacher {TEACHER} | {len(models)} runs:",
      ", ".join(m["run"] for m in models))
for run, other in left_out:
    print(f"left out: {run}, scored on the {other} export")


def line(m):
    return f"{m['wer']:6.2f} (S {m['sub']:.2f}  D {m['del']:.2f}  I {m['ins']:.2f})"


def label(m):
    return m["run"] + (" *" if m["run"] == TEACHER else "")

## Gold and val

Folded WER with S/D/I, raw WER, and the real-time factor of the decode on the GPU it was scored
on. `*` marks the teacher; the family is Flex, an ASR student or a speech-LLM student.

In [ ]:
print(f"{'model':<28}{'family':<12}{'epoch':>6}  {'val':<34}{'gold':<34}{'gold raw':>9}{'gold RTF':>10}")
for m in models:
    r = m["row"]
    print(f"{label(m):<28}{evalkit.family(r):<12}{str(r.get('best_epoch') or '-'):>6}  {line(r['val']):<34}"
          f"{line(r['gold']):<34}{r['gold']['raw_wer']:9.2f}{r['gold']['rtf']:10.4f}")
print("\ngold by crosstalk bucket, folded WER:")
print(f"{'model':<28}" + "".join(f"{b:>10}" for b in evalkit.BUCKETS))
for m in models:
    by = (m["row"].get("gold_by_class") or {}).get("overlap") or {}
    print(f"{label(m):<28}" + "".join(f"{by[b]['wer']:>10.2f}" if b in by else f"{'-':>10}" for b in evalkit.BUCKETS))

## Every model against the teacher

Model minus teacher in WER points, with a 95% interval from resampling episodes, on the clips both
scored: overall and per value of each class in `CLASSES`. An interval that contains zero, or lies
below it, meets the bar on that class.

In [ ]:
paired = {}
for m in models:
    if m["run"] == TEACHER:
        continue
    paired[m["run"]] = {}
    for split in ("val", "gold"):
        a, b = by_run[TEACHER]["counts"].get(split), m["counts"].get(split)
        if not a or not b:
            continue
        try:
            paired[m["run"]][split] = evalkit.pair(rows[split], a, b, keys=CLASSES)
        except ValueError:
            continue
for split in ("gold", "val"):
    keys = sorted({k for p in paired.values() for k in p.get(split, {}) if k != "clips"}, key=lambda k: (k != "all", k))
    print(f"\n{split}: model minus {TEACHER} [95% CI]")
    for run, p in paired.items():
        if split not in p:
            continue
        cells = "  ".join(f"{k} {p[split][k][0]:+.2f} [{p[split][k][1]:+.2f},{p[split][k][2]:+.2f}]"
                          for k in keys if k in p[split])
        print(f"  {run:<26} {cells}")

## What each stage added to a student, by family

Stage 2 minus stage 1 is the pseudo-labels together with the extra training (D106: stage 2
continues stage 1's weights, so the two cannot be told apart here). Stage 3 minus stage 2 is the
augmentation alone: both start from stage 1's weights. Grouped by family: whether the recipe
transfers to speech-LLM students is read here (D121).

In [ ]:
steps = {}
students = sorted({m["row"]["student"] for m in models if m["row"].get("student")},
                  key=lambda s: (evalkit.family({"student": s}), s))
for student in students:
    have = {m["row"]["stage"]: m for m in models if m["row"].get("student") == student}
    for before, after in (("human", "distill"), ("distill", "distill-aug")):
        if before not in have or after not in have:
            continue
        name = f"{student}: {after} minus {before}"
        steps[name] = {}
        for split in ("val", "gold"):
            a, b = have[before]["counts"].get(split), have[after]["counts"].get(split)
            if a and b:
                steps[name][split] = evalkit.pair(rows[split], a, b, keys=())
        cells = "  ".join(f"{split} {v['all'][0]:+.2f} [{v['all'][1]:+.2f},{v['all'][2]:+.2f}]"
                          for split, v in steps[name].items())
        print(f"{evalkit.family({'student': student}):<12}{name:<44} {cells}")
if not steps:
    print("no student has two stages scored yet")

## Student against student, stage by stage

Each pair in `CROSS`, at every stage both students have: A minus B in WER points, on the clips
both scored, with episodes resampled. Below zero, A is better. Omnilingual LLM-ASR minus
Omnilingual CTC is one encoder family with an LLM decoder against a CTC head (D121); a speech-LLM
student minus Whisper is D121's bar: worth its decode cost only below zero with an interval that
excludes it.

In [ ]:
cross = {}
for a, b in CROSS:
    for stage in ("human", "distill", "distill-aug"):
        ma, mb = by_run.get(f"{a}-{stage}"), by_run.get(f"{b}-{stage}")
        if ma is None or mb is None:
            continue
        name = f"{a} minus {b}, {stage}"
        cross[name] = {}
        for split in ("val", "gold"):
            of_b, of_a = mb["counts"].get(split), ma["counts"].get(split)
            if of_a and of_b:  # evalkit.pair gives WER(second) - WER(first): A minus B
                cross[name][split] = evalkit.pair(rows[split], of_b, of_a, keys=CLASSES)
        cells = "  ".join(f"{split} {v['all'][0]:+.2f} [{v['all'][1]:+.2f},{v['all'][2]:+.2f}]"
                          for split, v in cross[name].items())
        print(f"{name:<48} {cells}")
if not cross:
    print("no pair in CROSS has a stage scored for both students yet")

## The public sets

Folded WER per set and the mean over the sets, then each model minus base Flex with the set's own
unit resampled. For Flex, the difference to base is what fine-tuning cost outside our domain, and
what blending recovered. For a student that never knew Nepali there was nothing to forget: its
columns say how far the training generalises.

In [ ]:
SETS = tuple(evalkit.BENCHMARKS)


def set_files(m, name):
    # (summary, per-clip lines) of a model on a public set; None when it was not scored.
    remote = f"{m['prefix']}/{m['run']}/benchmarks/{name}.json"
    summary = fetch_json(m["repo"], remote)
    if summary is None:
        return None
    lines = Path(hf_hub_download(m["repo"], remote + "l", token=TOKEN)).read_text("utf-8").splitlines()
    return summary, [json.loads(x) for x in lines]


public = {m["run"]: {name: got for name in SETS if (got := set_files(m, name)) is not None} for m in models}
print(f"{'model':<28}" + "".join(f"{name:>14}" for name in SETS) + f"{'mean':>8}")
means = {}
for m in models:
    got = public[m["run"]]
    if not got:
        continue
    if len(got) == len(SETS):
        means[m["run"]] = sum(got[name][0]["wer"] for name in SETS) / len(SETS)
    print(f"{label(m):<28}" + "".join(f"{got[name][0]['wer']:>14.2f}" if name in got else f"{'-':>14}" for name in SETS)
          + (f"{means[m['run']]:>8.2f}" if m["run"] in means else f"{'-':>8}"))
vs_base = {}
if public.get("base"):
    print("\nmodel minus base [95% CI]")
    for m in models:
        if m["run"] == "base" or not public[m["run"]]:
            continue
        vs_base[m["run"]] = {name: evalkit.pair_benchmark(public["base"][name][1], public[m["run"]][name][1])["all"]
                             for name in SETS if name in public[m["run"]] and name in public["base"]}
        cells = "  ".join(f"{name} {d:+.2f} [{lo:+.2f},{hi:+.2f}]" for name, (d, lo, hi) in vs_base[m["run"]].items())
        print(f"  {label(m):<26} {cells}")
else:
    print("\nbase Flex has no public-set scores under this prefix: run 03b_Flex_Benchmarks.ipynb")

## Where the errors are

Blocks 1 and 2 of the breakdown (docs/architecture.md), per run and set, as each run's notebook wrote them beside
its WER (nothing is recomputed here): the WER of each crosstalk bucket with its share of the
errors, then the number errors' share and the WER over everything that is not a number. A run
scored before error mining has no breakdown and is left out of these tables. The rows behind them
are in each run's `harness/errors/`, which the Models page reads.

In [ ]:
BUCKET_COLUMNS = (*evalkit.BUCKETS, "unmeasured")
breakdowns = {}
for m in models:
    got = {}
    for split in ("gold", "val"):
        metrics = fetch_json(m["repo"], f"{m['prefix']}/{m['run']}/{split}_metrics.json") or {}
        if metrics.get("breakdown"):
            got[split] = metrics["breakdown"]
    for name, (summary, _) in public[m["run"]].items():
        if summary.get("breakdown"):
            got[name] = summary["breakdown"]
    breakdowns[m["run"]] = got
for set_name in ("gold", "val", *SETS):
    have = {run: b[set_name] for run, b in breakdowns.items() if set_name in b}
    if not have:
        print(f"\n{set_name}: no run has a breakdown (scored before error mining)")
        continue
    print(f"\n{set_name}: WER by crosstalk bucket (share of errors) | numbers: share of errors, WER without them")
    print(f"{'model':<28}{'WER':>8}" + "".join(f"{b:>16}" for b in BUCKET_COLUMNS) + f"{'numbers':>9}{'without':>9}")
    for run, b in have.items():
        buckets = {x["bucket"]: x for x in b["overlap"]}
        cells = "".join(f"{buckets[k]['wer']:>9.2f} ({100 * buckets[k]['share_of_errors']:>3.0f}%)" if k in buckets
                        else f"{'-':>16}" for k in BUCKET_COLUMNS)
        n = b["numbers"]
        print(f"{label(by_run[run]):<28}{b['wer']:>8.2f}{cells}{100 * n['share_of_errors']:>8.1f}%{n['wer_without']:>9.2f}")

## WER by word rarity (D123)

One table per model: a row per bucket of how often a word occurs in what the students train on
(the human train labels and the teacher's pseudo-labels, counted in the dataset's
`harness/word_counts.json`), a column per set, and in each cell the WER of that set's words in
that bucket, scored against the set's own labels: its errors per 100 of its reference words in
the bucket. A substitution and a deletion are the reference word's, an insertion the inserted
word's. A run scored before D123 has no rarity until `scripts/backfill_rarity.py` adds it.

In [ ]:
RARITY_SETS = ("val", "gold", *SETS)


rarity = {}
for m in models:
    got = {s: b["rarity"] for s in RARITY_SETS if (b := breakdowns[m["run"]].get(s)) and b.get("rarity")}
    if not got:
        continue
    rarity[m["run"]] = got
    names = [x["bucket"] for x in next(iter(got.values()))["buckets"]]
    columns = [s for s in RARITY_SETS if s in got]
    print(f"\n{label(m)}: WER by word rarity")
    print(f"{'bucket':<10}" + "".join(f"{s:>14}" for s in columns))
    for i, bucket in enumerate(names):
        cells = [got[s]["buckets"][i]["wer"] for s in columns]
        print(f"{bucket:<10}" + "".join(f"{'-':>14}" if c is None else f"{c:>14.2f}" for c in cells))
if not rarity:
    print("no run has a rarity block")

## Write the report

`report.json` holds every number above, the breakdowns included; `report.md` is the main tables as Markdown, to paste
into `docs/findings.md`.

In [ ]:
def cell(v):
    return f"{v[0]:+.2f} [{v[1]:+.2f}, {v[2]:+.2f}]"


md_lines = [f"Export {export['exported_at'][:10]}, {models[0]['card'].get('fold_version', '')}. "
            f"Teacher: `{TEACHER}` (marked `*`). Folded WER (S / D / I).", "",
            "| Model | Family | Val | Gold | Gold raw | Gold minus teacher | Public mean |", "|---|---|---|---|---|---|---|"]
for m in models:
    r = m["row"]
    vs = paired.get(m["run"], {}).get("gold")
    md_lines.append(
        f"| {label(m)} | {evalkit.family(r)} | {r['val']['wer']:.2f} ({r['val']['sub']:.2f} / {r['val']['del']:.2f} / {r['val']['ins']:.2f}) "
        f"| {r['gold']['wer']:.2f} ({r['gold']['sub']:.2f} / {r['gold']['del']:.2f} / {r['gold']['ins']:.2f}) "
        f"| {r['gold']['raw_wer']:.2f} | {cell(vs['all']) if vs else '—'} "
        f"| {format(means[m['run']], '.2f') if m['run'] in means else '—'} |")
md_lines += ["", "| Model | " + " | ".join(SETS) + " |", "|---|" + "---|" * len(SETS)]
for m in models:
    got = public[m["run"]]
    if got:
        md_lines.append(f"| {label(m)} | " + " | ".join(f"{got[n][0]['wer']:.2f}" if n in got else "—" for n in SETS) + " |")
for run, got in rarity.items():
    names = [x["bucket"] for x in next(iter(got.values()))["buckets"]]
    columns = [s for s in RARITY_SETS if s in got]
    md_lines += ["", f"WER by word rarity, {label(by_run[run])}", "",
                 "| Bucket | " + " | ".join(columns) + " |", "|---|" + "---|" * len(columns)]
    for i, bucket in enumerate(names):
        md_lines.append(f"| {bucket} | " + " | ".join(
            "—" if (c := got[s]["buckets"][i]["wer"]) is None else f"{c:.2f}" for s in columns) + " |")
for title, table in (("Step", steps), ("Pair", cross)):
    if table:
        md_lines += ["", f"| {title} | Val | Gold |", "|---|---|---|"]
        md_lines += [f"| {name} | " + " | ".join(cell(v[s]["all"]) if s in v else "—" for s in ("val", "gold")) + " |"
                     for name, v in table.items()]
report = {
    "export": export["exported_at"], "teacher": teacher,
    "models": [{"run": m["run"], "repo": m["repo"], "prefix": m["prefix"], "family": evalkit.family(m["row"]),
                "architecture": m["card"].get("architecture"), "row": m["row"]} for m in models],
    "left_out": left_out, "vs_teacher": paired, "stage_steps": steps, "cross": cross,
    "public": {run: {name: got[name][0] for name in got} for run, got in public.items()},
    "public_mean": means, "public_vs_base": vs_base, "breakdowns": breakdowns, "rarity": rarity,
}
name = distill.smoke_name(SMOKE, "report")  # a smoke run reads the real Flex folder; never overwrite its report
out = FT / "report"
out.mkdir(exist_ok=True)
(out / f"{name}.json").write_text(json.dumps(report, indent=1, ensure_ascii=False))
(out / f"{name}.md").write_text("\n".join(md_lines) + "\n", "utf-8")
api.upload_folder(repo_id=FLEX_REPO, folder_path=str(out), path_in_repo=FLEX_PREFIX,
                  commit_message=f"{FLEX_PREFIX}: report, {len(models)} runs")
print("\n".join(md_lines))
print("\nuploaded", f"https://huggingface.co/{FLEX_REPO}/tree/main/{FLEX_PREFIX}")